# SenGuard-MSC kernel D: ablations, panels, conformal sets, demo (G5)
Needs kernel B attached; its output carries everything from kernel A.

In [ ]:
# Restores the frozen outputs of kernel A at their original absolute paths, so hash checks stay meaningful.
import shutil
from pathlib import Path
hits = sorted(p for p in Path('/kaggle/input').rglob('frozen_model.json') if (p.parent / 'external_predictions.csv').exists())
if not hits:
    raise FileNotFoundError('Kernel B output is not attached: no frozen_model.json under /kaggle/input.')
src = hits[0].parents[1]
dst = Path('/kaggle/working/senguard_msc')
if not dst.exists():
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.git', '.venv_check'))
print('Restored', src, '->', dst, '|', sorted(p.name for p in dst.iterdir()))

# SenGuard-MSC · Notebook 05 — Research extension, ablations and demo (G5, G6 demo)

Development data picks everything here. Because leave-one-cell-type-out AUROC nearly saturates, a harder test is added: **leave-one-cell-type-and-paradigm-out**, where the test paradigm was never seen in training. MSC trends are shown only for choices already fixed on development data, and are labelled exploratory.

Small gene panels are an experimental hypothesis only. They are not a validated assay and no cost claim follows from them. Null results are reported the same way as positive ones.

In [ ]:
import json, sys, warnings
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = Path('/kaggle/working/senguard_msc') if Path('/kaggle/working/senguard_msc').exists() else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, DEV, ASC, BM
from joblib import Parallel, delayed
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
sg.require(PROC / f'{DEV}_labels.csv', RESULTS / 'frozen_model.json', RESULTS / 'external_predictions.csv')
labels = pd.read_csv(PROC / f'{DEV}_labels.csv').set_index('sample_id')
labels['paradigm'] = labels.treatment.str.extract(r'^(\w+)', expand=False).replace({'Control': 'control', 'Empty': 'control'})
counts, _ = sg.load_as_deposited(DEV)
counts = counts.loc[labels.index]
y, groups, paradigm = labels.y.to_numpy(), labels.cell_line.to_numpy(), labels.paradigm.to_numpy()
print(pd.crosstab(labels.cell_line, labels.paradigm).to_string())
ens2sym = pd.read_csv(PROC / 'GSE309666_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
hall, extra = sg.load_gene_sets()
N_JOBS = -1

## Shared evaluation
Each candidate is a function `(Xtr_counts, ytr, Xte_counts) -> test decision values`, fit on training samples only. Two designs:
- **LOCO:** hold out a cell type, score against its controls.
- **LOCPO:** hold out a cell type and also remove the test paradigm's positives from training. Each (cell type, paradigm) pair is one result, compared with that cell type's controls.

In [ ]:
def loco_splits():
    for g in np.unique(groups):
        te = np.flatnonzero(groups == g)
        tr = np.flatnonzero(groups != g)
        yield {'design': 'LOCO', 'cell_type': g, 'paradigm': 'all', 'tr': tr, 'te': te}

def locpo_splits():
    for g in np.unique(groups):
        for p in sorted(set(paradigm[(groups == g) & (y == 1)])):
            te = np.flatnonzero((groups == g) & ((paradigm == p) | (y == 0)))
            tr = np.flatnonzero((groups != g) & (paradigm != p))
            yield {'design': 'LOCPO', 'cell_type': g, 'paradigm': p, 'tr': tr, 'te': te}

SPLITS = list(loco_splits()) + list(locpo_splits())
print('LOCO folds', sum(s['design'] == 'LOCO' for s in SPLITS), '| LOCPO folds', sum(s['design'] == 'LOCPO' for s in SPLITS))

def norm_pair(ctr, cte, how):
    genes, ref = sg.prefilter_train(ctr)
    if how == 'mor':
        return sg.fold_transform(ctr, genes, ref), sg.fold_transform(cte, genes, ref)
    if how == 'none':
        return np.log1p(ctr[genes]), np.log1p(cte[genes])
    if how == 'cpm':
        return np.log1p(sg.scale_to_million(ctr)[genes]), np.log1p(sg.scale_to_million(cte)[genes])
    if how == 'rank':
        return ctr[genes].rank(axis=1, pct=True), cte[genes].rank(axis=1, pct=True)
    raise ValueError(how)

def top_k(Xtr, ytr, k):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        keep = Xtr.std() > 0
        sel = sg.SelectKBest(sg.f_classif, k=min(k, int(keep.sum()))).fit(Xtr.loc[:, keep].to_numpy(), ytr)
    return Xtr.columns[keep][sel.get_support(indices=True)]

def lr_gene(ctr, ytr, cte, k=5000, how='mor', C=1.0, penalty='l2'):
    Xtr, Xte = norm_pair(ctr, cte, how)
    use = top_k(Xtr, ytr, k)
    ss = StandardScaler().fit(Xtr[use])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m = LogisticRegression(solver='liblinear', C=C, penalty=penalty, max_iter=2000, random_state=sg.SEED).fit(ss.transform(Xtr[use]), ytr)
    return m.decision_function(ss.transform(Xte[use])), list(use)

def symbols_of(c):
    return sg.to_symbols(c, pd.DataFrame({'symbol': ens2sym.reindex(c.columns)}, index=c.columns))

def pathway_features(c):
    sym = symbols_of(c)
    ranks = sym.rank(axis=1, pct=True)
    return pd.DataFrame({k: ranks[[g for g in v if g in ranks.columns]].mean(axis=1) for k, v in hall.items()})

PW = pathway_features(counts)
HUSI = sg.husi_raw(symbols_of(counts), sg.load_husi_weights())[0]

def lr_pathway(tr, te, ytr, C=0.1):
    ss = StandardScaler().fit(PW.iloc[tr])
    m = LogisticRegression(solver='liblinear', C=C, max_iter=2000, random_state=sg.SEED).fit(ss.transform(PW.iloc[tr]), ytr)
    return m.decision_function(ss.transform(PW.iloc[te])), m

def run_split(s, name, fn):
    tr, te = s['tr'], s['te']
    score = fn(tr, te)
    return {'design': s['design'], 'cell_type': s['cell_type'], 'paradigm': s['paradigm'], 'candidate': name, 'auroc': sg.roc_auc_score(y[te], score)}

def evaluate(candidates):
    out = Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(run_split)(s, n, f) for n, f in candidates.items() for s in SPLITS)
    return pd.DataFrame(out)

def summarize(res):
    rows = []
    for (cand, design), g in res.groupby(['candidate', 'design']):
        m, lo, hi = sg.bootstrap_mean_ci(g.groupby('cell_type').auroc.mean())
        rows.append({'candidate': cand, 'design': design, 'mean_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_auroc': g.auroc.min(), 'n_results': len(g)})
    return pd.DataFrame(rows).pivot(index='candidate', columns='design', values=['mean_auroc', 'ci_low', 'ci_high', 'min_auroc'])

## Ablation 1: normalization [measured]
Same model, four ways of scaling. Rank is included because it doesn't depend on platform scale.

In [ ]:
cands = {f'L2 genes, {h}': (lambda tr, te, h=h: lr_gene(counts.iloc[tr], y[tr], counts.iloc[te], how=h)[0]) for h in ['mor', 'none', 'cpm', 'rank']}
abl_norm = evaluate(cands)
abl_norm.to_csv(RESULTS / 'g5_ablation_normalization.csv', index=False)
print(summarize(abl_norm).round(3).to_string())

## Ablation 2: feature families and fusion [measured]
Gene models, Hallmark pathway scores, fixed scores that fit nothing, and an equal-weight late fusion. Fusion converts each member to z-scores with training-sample statistics, using training scores from the same fold.

In [ ]:
def fusion(tr, te):
    Xtr, Xte = norm_pair(counts.iloc[tr], counts.iloc[te], 'mor')
    use = top_k(Xtr, y[tr], 5000)
    ss = StandardScaler().fit(Xtr[use])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        g = LogisticRegression(solver='liblinear', max_iter=2000, random_state=sg.SEED).fit(ss.transform(Xtr[use]), y[tr])
    gt, ge = g.decision_function(ss.transform(Xtr[use])), g.decision_function(ss.transform(Xte[use]))
    pe, pm = lr_pathway(tr, te, y[tr])
    pt = pm.decision_function(StandardScaler().fit(PW.iloc[tr]).transform(PW.iloc[tr]))
    ht, he = HUSI.iloc[tr].to_numpy(), HUSI.iloc[te].to_numpy()
    z = lambda a, ref: (a - ref.mean()) / ref.std()
    return (z(ge, gt) + z(pe, pt) + z(he, ht)) / 3

E2F = pd.Series(sg.rank_set_score(symbols_of(counts), hall['HALLMARK_E2F_TARGETS'])[0], index=counts.index)
SENMAYO = pd.Series(sg.rank_set_score(symbols_of(counts), extra['SenMayo'])[0], index=counts.index)
cands = {'L2 genes (k=5000)': lambda tr, te: lr_gene(counts.iloc[tr], y[tr], counts.iloc[te])[0],
         'L1 genes (k=2000)': lambda tr, te: lr_gene(counts.iloc[tr], y[tr], counts.iloc[te], k=2000, C=0.1, penalty='l1')[0],
         'Hallmark pathway LR': lambda tr, te: lr_pathway(tr, te, y[tr])[0],
         'hUSI raw (fixed)': lambda tr, te: HUSI.iloc[te].to_numpy(),
         'SenMayo (fixed)': lambda tr, te: SENMAYO.iloc[te].to_numpy(),
         'Hallmark E2F inverted (fixed)': lambda tr, te: -E2F.iloc[te].to_numpy(),
         'Late fusion: L2 + pathway + hUSI': fusion}
abl_fam = evaluate(cands)
abl_fam.to_csv(RESULTS / 'g5_ablation_families.csv', index=False)
fam = summarize(abl_fam)
print(fam.round(3).to_string())
a = abl_fam[abl_fam.candidate == 'L2 genes (k=5000)'].set_index(['design', 'cell_type', 'paradigm']).auroc
b = abl_fam[abl_fam.candidate == 'Late fusion: L2 + pathway + hUSI'].set_index(['design', 'cell_type', 'paradigm']).auroc
d = (b - a).groupby(level=['design', 'cell_type']).mean().groupby(level='design')
print('\nFusion minus L2 genes, mean over cell types:', d.mean().round(4).to_dict(), '| cell types improved / worsened:', {k: (int((v > 0).sum()), int((v < 0).sum())) for k, v in d})
print('\nHardest paradigms under LOCPO (mean AUROC):')
print(abl_fam[abl_fam.design == 'LOCPO'].pivot_table(index='paradigm', columns='candidate', values='auroc').round(3).to_string())

## Ablation 3: model complexity and small panels [measured]
Panels are chosen inside each fold, by univariate rank or by L1, so these estimates include selection. Stability is the Jaccard overlap of the chosen genes across LOCO folds.

In [ ]:
def l1_panel(Xtr, ytr, k):
    pre = top_k(Xtr, ytr, 2000)
    ss = StandardScaler().fit(Xtr[pre])
    Z = ss.transform(Xtr[pre])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for C in np.geomspace(0.002, 2, 25):
            m = LogisticRegression(solver='liblinear', penalty='l1', C=C, max_iter=2000, random_state=sg.SEED).fit(Z, ytr)
            if (m.coef_[0] != 0).sum() >= k:
                break
    order = np.argsort(-np.abs(m.coef_[0]))[:k]
    return list(pre[order])

def panel_fit(ctr, ytr, cte, k, method):
    Xtr, Xte = norm_pair(ctr, cte, 'mor')
    use = list(top_k(Xtr, ytr, k)) if method == 'univariate' else l1_panel(Xtr, ytr, k)
    ss = StandardScaler().fit(Xtr[use])
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=2000, random_state=sg.SEED).fit(ss.transform(Xtr[use]), ytr)
    return m.decision_function(ss.transform(Xte[use])), use

SIZES = [10, 20, 30, 100, 500, 2000, 5000]
def one(s, k, method):
    score, use = panel_fit(counts.iloc[s['tr']], y[s['tr']], counts.iloc[s['te']], k, method)
    return {'design': s['design'], 'cell_type': s['cell_type'], 'paradigm': s['paradigm'], 'k': k, 'method': method, 'auroc': sg.roc_auc_score(y[s['te']], score), 'genes': use if s['design'] == 'LOCO' else None}
jobs = [(s, k, 'univariate') for s in SPLITS for k in SIZES] + [(s, k, 'l1') for s in SPLITS for k in [10, 20, 30]]
panel_res = pd.DataFrame(Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(one)(*j) for j in jobs))
panel_res.drop(columns='genes').to_csv(RESULTS / 'g5_panels_by_fold.csv', index=False)
rows = []
for (method, k, design), g in panel_res.groupby(['method', 'k', 'design']):
    m, lo, hi = sg.bootstrap_mean_ci(g.groupby('cell_type').auroc.mean())
    row = {'method': method, 'k': k, 'design': design, 'mean_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_auroc': g.auroc.min()}
    if design == 'LOCO':
        sets = [set(x) for x in g.genes]
        row['jaccard_across_folds'] = float(np.mean([len(a & b) / len(a | b) for a, b in combinations(sets, 2)]))
    rows.append(row)
panel_sum = pd.DataFrame(rows).sort_values(['method', 'design', 'k'])
panel_sum.to_csv(RESULTS / 'g5_panel_summary.csv', index=False)
print(panel_sum.round(3).to_string(index=False))

## Frozen experimental panels, from development data only
Genes are ranked by how often they were chosen across LOCO folds (L1, k=30), then by mean absolute coefficient on all development data. Sizes 10, 20 and 30 are saved before the MSC check below. The proliferation share shows how much of a panel is plain cell-cycle biology.

In [ ]:
sel = panel_res[(panel_res.method == 'l1') & (panel_res.k == 30) & (panel_res.design == 'LOCO')]
freq = pd.Series([g for x in sel.genes for g in x]).value_counts()
Xall, _ = norm_pair(counts, counts, 'mor')
dropped = [g for g in freq.index if g not in Xall.columns]
print(f'{len(dropped)} of {len(freq)} fold-selected genes do not pass the gene filter on all development data and are excluded from ranking: {dropped}')
freq = freq[[g for g in freq.index if g in Xall.columns]]
ssa = StandardScaler().fit(Xall[freq.index])
m_all = LogisticRegression(solver='liblinear', C=1.0, max_iter=2000, random_state=sg.SEED).fit(ssa.transform(Xall[freq.index]), y)
panel_tbl = pd.DataFrame({'gene_id': freq.index, 'symbol': ens2sym.reindex(freq.index).values, 'folds_selected': freq.values, 'abs_coef_all_dev': np.abs(m_all.coef_[0])}).sort_values(['folds_selected', 'abs_coef_all_dev'], ascending=False).reset_index(drop=True)
prolif = set(hall['HALLMARK_E2F_TARGETS']) | set(hall['HALLMARK_G2M_CHECKPOINT'])
pub = sg.load_sencat_markers()
panel_tbl['is_e2f_g2m'] = panel_tbl.symbol.isin(prolif)
panel_tbl['in_senmayo'] = panel_tbl.symbol.isin(extra['SenMayo'])
panel_tbl['published_sencat_coef'] = panel_tbl.gene_id.map(pub.coef)
panel_tbl.to_csv(RESULTS / 'g5_panel_ranked.csv', index=False)
print(panel_tbl.head(30).round(3).to_string(index=False))
PANELS = {k: panel_tbl.gene_id.head(k).tolist() for k in [10, 20, 30]}
(RESULTS / 'g5_panels_frozen.json').write_text(json.dumps(PANELS, indent=1))
print('\nPanel composition:', {k: {'e2f_g2m_share': float(panel_tbl.is_e2f_g2m.head(k).mean()), 'senmayo_share': float(panel_tbl.in_senmayo.head(k).mean())} for k in PANELS})

## Exploratory MSC check of frozen panels
Weights are refit on development cell types excluding the target lineage, and panels stay as frozen above. This is the only use of MSC data in this notebook and it selected nothing. Per-gene direction across donors is also reported for the 30-gene panel, to expose donor-specific effects.

In [ ]:
manifest = sg.load_manifest()
asc_meta = manifest[(manifest.study_accession == ASC) & manifest.use_in_mrna_analysis]
bm_meta = manifest[(manifest.study_accession == BM) & manifest.use_in_mrna_analysis]
asc_norm = pd.read_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[asc_meta.index]
bm_tpm = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[bm_meta.index]
rows = []
for cohort, x, meta, subj, drop in [(ASC, asc_norm, asc_meta, 'donor', ['BMMSC', 'prAD']), (BM, bm_tpm, bm_meta, 'candidate_biological_id', ['BMMSC'])]:
    mask = ~np.isin(groups, drop)
    Xd, _ = norm_pair(counts.loc[mask], counts.loc[mask], 'mor')
    for k, genes in PANELS.items():
        have = [g for g in genes if g in x.columns and g in Xd.columns]
        ss = StandardScaler().fit(Xd[have])
        w = LogisticRegression(solver='liblinear', C=1.0, max_iter=2000, random_state=sg.SEED).fit(ss.transform(Xd[have]), y[mask]).coef_[0] / ss.scale_
        s = pd.Series(np.log1p(x[have]).to_numpy() @ w, index=x.index)
        t = sg.per_donor_trend(s, meta.passage, meta[subj].astype(str))
        rows.append({'cohort': cohort, 'panel_size': k, 'genes_present': len(have), 'mean_slope': t.slope_per_passage.mean(), 'positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t), 'sign_flip_p': sg.exact_sign_flip_p(t.slope_per_passage), 'standardized_slope': t.slope_per_passage.mean() / s.std()})
panel_ext = pd.DataFrame(rows)
panel_ext.to_csv(RESULTS / 'g5_panels_external_exploratory.csv', index=False)
print(panel_ext.round(3).to_string(index=False))
gene_rows = []
logx = np.log1p(asc_norm)
for g in PANELS[30]:
    if g in logx.columns:
        t = sg.per_donor_trend(logx[g], asc_meta.passage, asc_meta.donor)
        gene_rows.append({'gene_id': g, 'symbol': ens2sym.get(g), 'donors_up': int((t.slope_per_passage > 0).sum()), 'donors_down': int((t.slope_per_passage < 0).sum()), 'dev_sign': np.sign(pub.coef.get(g, np.nan))})
gene_dir = pd.DataFrame(gene_rows)
gene_dir.to_csv(RESULTS / 'g5_panel_gene_direction_asc.csv', index=False)
print(gene_dir.to_string(index=False))

## Uncertainty: Mondrian conformal sets by held-out cell type [measured]
Inside each outer fold, out-of-fold scores come from inner leave-one-cell-type-out, so calibration cell types differ from those used to fit the inner model. Class-conditional conformal quantiles give each held-out sample a set: `{0}`, `{1}`, both (abstain), or neither (out of distribution).

Coverage assumes exchangeability between cell types, which can't be assumed for MSC passage samples, so **no sets are produced for the external cohorts**. Replicates inside a cell line also aren't independent, so coverage should be read per cell type.

In [ ]:
def conformal_fold(g, alpha=0.1):
    tr, te = np.flatnonzero(groups != g), np.flatnonzero(groups == g)
    oof = np.full(len(tr), np.nan)
    for h in np.unique(groups[tr]):
        i = groups[tr] == h
        oof[i] = lr_gene(counts.iloc[tr[~i]], y[tr][~i], counts.iloc[tr[i]], k=2000)[0]
    s_te = lr_gene(counts.iloc[tr], y[tr], counts.iloc[te], k=2000)[0]
    q = {}
    for c, sign in [(1, -1), (0, 1)]:
        cal = sign * oof[y[tr] == c]
        n = len(cal)
        q[c] = np.quantile(cal, min(1, np.ceil((n + 1) * (1 - alpha)) / n), method='higher')
    rows = []
    for i, v in zip(te, s_te):
        in1, in0 = -v <= q[1], v <= q[0]
        rows.append({'sample_id': counts.index[i], 'cell_type': g, 'y': int(y[i]), 'score': float(v), 'in_set_1': bool(in1), 'in_set_0': bool(in0)})
    return rows
conf = pd.DataFrame([r for rows in Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(conformal_fold)(g) for g in np.unique(groups)) for r in rows])
conf['covered'] = np.where(conf.y == 1, conf.in_set_1, conf.in_set_0)
conf['set_type'] = np.select([conf.in_set_1 & conf.in_set_0, conf.in_set_1, conf.in_set_0], ['both_abstain', 'senescent_like', 'control_like'], 'neither_out_of_distribution')
conf.to_csv(RESULTS / 'g5_conformal_sets.csv', index=False)
by_ct = conf.groupby('cell_type').agg(n=('covered', 'size'), coverage=('covered', 'mean'), abstain=('set_type', lambda s: (s == 'both_abstain').mean()), neither=('set_type', lambda s: (s.str.startswith('neither')).mean())).round(3)
print(by_ct.to_string())
print('\nOverall coverage %.3f (target 0.90), mean over cell types %.3f, min cell type %.3f' % (conf.covered.mean(), by_ct.coverage.mean(), by_ct.coverage.min()))
print(conf.set_type.value_counts().to_string())

## Figure 5: panel size and the harder test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1, 1.15]})
ax = axes[0]
u = panel_sum[panel_sum.method == 'univariate']
for design, color, label in [('LOCO', sg.PAL['blue'], 'Held-out cell type'), ('LOCPO', sg.PAL['orange'], 'Held-out cell type and paradigm')]:
    g = u[u.design == design].sort_values('k')
    ax.fill_between(g.k, g.ci_low, g.ci_high, color=color, alpha=0.12, linewidth=0)
    ax.plot(g.k, g.mean_auroc, color=color, marker='o', ms=6, mec=sg.PAL['surface'], mew=1.5, label=label)
ax.set_xscale('log'); ax.set_xlabel('Genes chosen inside each fold'); ax.set_ylabel('Mean AUROC, 95% CI over cell types'); ax.set_title('Panel size'); ax.legend(fontsize=8, loc='lower right'); ax.grid(axis='x', visible=False)
ax = axes[1]
order = ['SenMayo (fixed)', 'Hallmark E2F inverted (fixed)', 'hUSI raw (fixed)', 'Hallmark pathway LR', 'L1 genes (k=2000)', 'L2 genes (k=5000)', 'Late fusion: L2 + pathway + hUSI']
s = abl_fam[abl_fam.design == 'LOCPO'].groupby(['candidate', 'cell_type']).auroc.mean().unstack(0)[order]
yy = np.arange(len(order))
for i, c in enumerate(order):
    m, lo, hi = sg.bootstrap_mean_ci(s[c])
    ax.plot([lo, hi], [i, i], color=sg.PAL['blue'], lw=2, solid_capstyle='round')
    ax.scatter([m], [i], s=40, color=sg.PAL['blue'], edgecolors=sg.PAL['surface'], linewidths=1.5, zorder=3)
ax.axvline(0.5, color=sg.PAL['muted'], lw=1, ls=(0, (3, 3)))
ax.set_yticks(yy, order, fontsize=8.5); ax.set_xlabel('Mean AUROC, cell type and paradigm held out'); ax.set_title('Feature families'); ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(sg.FIGS / 'fig5_panels_and_families.png')
plt.show()

## Demo: inspect one real sample
Choose a sample and see its measured values, published score, the out-of-fold model result if the sample is in the development set, and its donor's trajectory if it is an MSC. Every line says what kind of evidence it is. In an interactive session the dropdown appears; a batch run shows two default samples.

There is no calibrated probability for MSC samples, because no labels exist to calibrate one against. Nothing here says anything about potency, immune function or safety.

In [ ]:
GENES = ['CDKN1A', 'CDKN2A', 'GDF15', 'IL6', 'LMNB1', 'MKI67', 'TOP2A']
EXT_PRED = pd.read_csv(RESULTS / 'external_predictions.csv')
DEV_PRED = pd.read_csv(RESULTS / 'dev_nested_loco_predictions.csv')
FREEZE = json.loads((RESULTS / 'frozen_model.json').read_text())
dev_sym = symbols_of(pd.read_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[labels.index])
dev_pw = PW
asc_sym = symbols_of(asc_norm)
PUB_DEV = sg.weighted_log1p_score(pd.read_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[labels.index], pub.coef)[0]

def show_sample(sample_id):
    print('=' * 92)
    if sample_id in labels.index:
        r = labels.loc[sample_id]
        print(f'{sample_id} | SenCat | {r.cell_line} | {r.treatment}\n[GEO observation] condition label: {"senescence-induced" if r.y else "control"}')
        print('[measured, log1p normalized counts]', {g: round(float(np.log1p(dev_sym.loc[sample_id, g])), 2) for g in GENES if g in dev_sym.columns})
        print('[measured, rank score] Hallmark E2F %.3f | G2M %.3f' % (dev_pw.loc[sample_id, 'HALLMARK_E2F_TARGETS'], dev_pw.loc[sample_id, 'HALLMARK_G2M_CHECKPOINT']))
        print('[inference] published SenCat score %.2f; weights were fit with this sample, so read it with care' % PUB_DEV[sample_id])
        p = DEV_PRED[(DEV_PRED.model == FREEZE['chosen_model']) & (DEV_PRED.sample_id == sample_id)].iloc[0]
        print(f"[inference] {FREEZE['chosen_model']} with this cell type held out: decision value {p.score:.2f}, training-fold threshold {p.threshold_from_training:.2f}, call {'senescent-like' if p.call else 'control-like'}. Valid only for this induced-senescence setting.")
        c = pd.read_csv(RESULTS / 'g5_conformal_sets.csv').set_index('sample_id').loc[sample_id]
        print(f'[inference] conformal set at nominal 90%: {c.set_type}')
        same = labels[labels.cell_line == r.cell_line]
        fig, ax = plt.subplots(figsize=(6, 2.6))
        ax.scatter(PUB_DEV[same.index], np.where(same.y == 1, 1, 0) + np.random.default_rng(0).uniform(-.08, .08, len(same)), s=40, color=sg.PAL['muted'], linewidths=0)
        ax.scatter([PUB_DEV[sample_id]], [r.y], s=90, color=sg.PAL['orange'], edgecolors=sg.PAL['surface'], linewidths=1.5, zorder=3)
        ax.set_yticks([0, 1], ['Control', 'Induced']); ax.set_xlabel('Published SenCat score'); ax.set_title(f'{sample_id} among {r.cell_line} samples'); ax.grid(axis='y', visible=False)
        plt.show()
    else:
        r = manifest.loc[sample_id]
        e = EXT_PRED[EXT_PRED.sample_id == sample_id]
        subject = e.subject.iloc[0]
        print(f'{sample_id} | {r.study_accession} | {r.title}\n[GEO observation] donor or id {subject}, passage {int(r.passage)}. No per-sample senescence label exists.')
        if r.study_accession == ASC and sample_id in asc_sym.index:
            print('[measured, log1p normalized counts]', {g: round(float(np.log1p(asc_sym.loc[sample_id, g])), 2) for g in GENES if g in asc_sym.columns})
            print(f'[published, per donor] growth status: {r.published_donor_growth_status}')
        s = e[e.score_name == 'SenCat published'].iloc[0].score
        print(f'[inference] published SenCat score {s:.2f}. No threshold is applied, because none can be justified across platforms.')
        tr = EXT_PRED[(EXT_PRED.score_name == 'SenCat published') & (EXT_PRED.subject == subject) & (EXT_PRED.study_accession == r.study_accession)].sort_values('passage')
        fig, ax = plt.subplots(figsize=(5.5, 2.8))
        ax.plot(tr.passage, tr.score, color=sg.PAL['blue'], marker='o', ms=6, mec=sg.PAL['surface'], mew=1.5)
        ax.scatter([r.passage], [s], s=90, color=sg.PAL['orange'], edgecolors=sg.PAL['surface'], linewidths=1.5, zorder=3)
        ax.set_xlabel('Passage'); ax.set_ylabel('SenCat score'); ax.set_title(f'{subject}: within-donor trajectory (inference)'); ax.grid(axis='x', visible=False)
        plt.show()
    print('Not a measure of potency, immune function or clinical safety.')

try:
    import ipywidgets as w
    from IPython.display import display
    options = sorted(EXT_PRED.sample_id.unique()) + labels.index[labels.cell_line.isin(['BMMSC', 'prAD'])].tolist()
    picker = w.Dropdown(options=options, description='Sample')
    out = w.Output()
    def _on(change):
        with out:
            out.clear_output(wait=True)
            show_sample(change['new'])
    picker.observe(_on, names='value')
    display(picker, out)
except Exception as e:
    print('Widgets unavailable:', e)
show_sample(labels.index[(labels.cell_line == 'BMMSC') & (labels.y == 1)][0])
show_sample(asc_meta.index[asc_meta.passage == asc_meta.passage.max()][0])

## G5 report

In [ ]:
report = f"""# G5: research extension and ablations

All choices come from development data. MSC results below selected nothing and are exploratory.

## Normalization [measured]
{summarize(abl_norm).round(3).to_string()}

## Feature families and fusion [measured]
{fam.round(3).to_string()}

Fusion minus L2 genes, mean over cell types: {d.mean().round(4).to_dict()}.

## Panel size [measured]
{panel_sum.round(3).to_string(index=False)}

## Frozen experimental panels
Top genes are in results/g5_panel_ranked.csv. These are hypotheses, not a validated assay, and no cost claim follows.

## Exploratory MSC check of panels
{panel_ext.round(3).to_string(index=False)}

## Conformal sets on held-out cell types [measured]
Overall coverage {conf.covered.mean():.3f} at target 0.90; lowest cell type {by_ct.coverage.min():.3f}. Sets: {conf.set_type.value_counts().to_dict()}. No sets are produced for MSC passage samples.
"""
(REPORTS / 'G5_ABLATIONS.md').write_text(report, encoding='utf-8')
print(report)
print(sg.output_manifest([RESULTS / f for f in ['g5_ablation_normalization.csv', 'g5_ablation_families.csv', 'g5_panels_by_fold.csv', 'g5_panel_summary.csv', 'g5_panel_ranked.csv', 'g5_panels_frozen.json', 'g5_panels_external_exploratory.csv', 'g5_panel_gene_direction_asc.csv', 'g5_conformal_sets.csv']] + [sg.FIGS / 'fig5_panels_and_families.png', REPORTS / 'G5_ABLATIONS.md']).assign(path=lambda t: t.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())